# Enterprise HR AI Copilot — Production RAG Architecture
### Complete Flow: Document Ingestion -> Hugging Face Embeddings -> Pinecone Vector Store -> Groq Standard RAG -> Improving RAG

This notebook demonstrates our modular, production-grade RAG architecture.
All business logic is cleanly decoupled into modular Python packages under `app/`:

| Module | File | Purpose |
| :--- | :--- | :--- |
| **Config** | `app/config.py` | Typed settings loading API keys and model configurations from `.env` |
| **Schemas** | `app/schemas.py` | Pydantic data contracts (`DocumentChunk`, `Citation`, `RAGResponse`) |
| **Ingestion** | `app/ingestion/` | Multi-format loader (PDF/DOCX/TXT), text cleaner, and recursive chunker |
| **Embeddings**| `app/embeddings.py` | Hugging Face API client (`BAAI/bge-large-en-v1.5`, 1024 dimensions) |
| **Vector Store** | `app/vectorstore.py` | Pinecone vector store wrapper for upserts and cosine similarity search |
| **Standard RAG** | `app/rag.py` | Groq LLM integration (`qwen/qwen3.8-27b`) for grounded answers |
| **Improving RAG**| `app/advanced_rag.py` | Query Rewriting, Evidence Grading (noise filtering), and Hallucination Checks |


In [1]:
# ==============================================================================
# Cell 1: Environment Setup & Module Imports
# ==============================================================================
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import configuration
from app.config import settings

# Import core modules
from app.schemas import DocumentChunk, RAGResponse, Citation
from app.ingestion import ingest_document
from app.embeddings import HuggingFaceEmbeddingClient
from app.vectorstore import PineconeVectorStore
from app.rag import StandardRAG, GroqLLMClient
from app.advanced_rag import AdvancedRAG, QueryRewriter, EvidenceGrader

print("[x] Configuration Loaded:")
print(f"  - Pinecone Index   : {settings.index_name}")
print(f"  - Embedding Model  : {settings.EMBEDDING_MODEL} ({settings.EMBEDDING_DIMENSION} dims)")
print(f"  - Groq LLM Model   : {settings.GROQ_MODEL}")
print(f"  - HF Token Present : {bool(settings.HUGGINGFACEHUB_API_TOKEN)}")
print(f"  - Groq Key Present : {bool(settings.GROQ_API_KEY)}")


[x] Configuration Loaded:
  - Pinecone Index   : ragbot
  - Embedding Model  : BAAI/bge-large-en-v1.5 (1024 dims)
  - Groq LLM Model   : qwen/qwen3.8-27b
  - HF Token Present : True
  - Groq Key Present : True


---
## Section 1: Modular Document Ingestion (`app.ingestion`)

In Phase 1, we established our ingestion pipeline. In our production package, this is encapsulated in `app/ingestion/`:
1. `loader.py`: Extracts page-aware text from PDF, DOCX, and TXT files.
2. `cleaner.py`: Strips unicode noise, repairs broken hyphenated linebreaks, removes recurring headers, and collapses whitespace.
3. `chunker.py`: Splits text using `RecursiveCharacterTextSplitter` while preserving exact page provenance.
4. `pipeline.py`: Assembles everything into `ingest_document()`.


In [2]:
# ==============================================================================
# Cell 2: Ingesting HR Policy Document
# ==============================================================================
pdf_path = PROJECT_ROOT / "uploads" / "ABC_Company_Leave_Policy.pdf"
chunks = ingest_document(pdf_path)

print(f"[OK] Ingestion Completed for: {pdf_path.name}")
print(f"  - Total Chunks Produced : {len(chunks)}")
print(f"  - Document ID           : {chunks[0].doc_id}")
print(f"  - Avg Chunk Size (chars): {sum(c.metadata.char_count for c in chunks) // len(chunks)}")
print(f"  - Page Distribution     : {dict((p, len([c for c in chunks if c.metadata.page_number == p])) for p in sorted(set(c.metadata.page_number for c in chunks)))}")

print(f"\n--- Sample Chunk 0 (Page {chunks[0].metadata.page_number}) ---")
print(f"ID: {chunks[0].chunk_id}")
print(f"Content:\n{chunks[0].content[:280]}...")


[OK] Ingestion Completed for: ABC_Company_Leave_Policy.pdf
  - Total Chunks Produced : 19
  - Document ID           : b2d60734cc88
  - Avg Chunk Size (chars): 598
  - Page Distribution     : {1: 5, 2: 6, 3: 6, 4: 2}

--- Sample Chunk 0 (Page 1) ---
ID: b2d60734cc88_p1_c000
Content:
ABC COMPANY
Employee Leave Policy
Document No: ABC-HR-POL-004 | Version: 2.3 | Effective Date: 01 April 2026 | Department: Human Resources
1. Purpose and Scope
This Leave Policy establishes the framework governing all categories of employee leave at ABC Company ("the
Company"). I...


---
## Section 2: Embeddings with Hugging Face API (`app.embeddings`)

We use `app.embeddings.HuggingFaceEmbeddingClient` to generate dense vector representations:
- **Model**: `BAAI/bge-large-en-v1.5`
- **Dimensions**: **1024**, aligning with our Pinecone index `ragbot`.
- **Infrastructure**: Hugging Face Router Inference API (`router.huggingface.co`) with automatic retry and batching.


In [3]:
# ==============================================================================
# Cell 3: Generating Vector Embeddings via Hugging Face API
# ==============================================================================
hf_client = HuggingFaceEmbeddingClient()

# 1. Embed document chunks in batches
chunk_texts = [c.content for c in chunks]
print(f"Generating embeddings for {len(chunk_texts)} chunks...")
chunk_embeddings = hf_client.embed_documents(chunk_texts, batch_size=16)

print(f"[OK] Generated {len(chunk_embeddings)} vector embeddings.")
print(f"  - Vector Dimensionality: {len(chunk_embeddings[0])} dimensions (matches Pinecone)")

# 2. Test query embedding
sample_query = "How many days of bereavement leave are granted?"
query_vector = hf_client.embed_query(sample_query)
print(f"[OK] Query embedding generated: length={len(query_vector)}, sample={query_vector[:4]}")


Generating embeddings for 19 chunks...
[OK] Generated 19 vector embeddings.
  - Vector Dimensionality: 1024 dimensions (matches Pinecone)
[OK] Query embedding generated: length=1024, sample=[0.04147305339574814, 0.00866425409913063, -0.028116218745708466, 0.01107439398765564]


---
## Section 3: Vector Store with Pinecone (`app.vectorstore`)

`app.vectorstore.PineconeVectorStore` handles vector storage and indexing:
- Connects securely to Pinecone via `PINECONE_API_KEY`.
- Target index: `ragbot` (1024-dim, cosine distance).
- **Metadata Storage**: Stores the full text content, source filename, and page number directly inside each vector's metadata payload for single-roundtrip retrieval.


In [4]:
# ==============================================================================
# Cell 4: Upserting Chunks & Semantic Vector Search in Pinecone
# ==============================================================================
vector_store = PineconeVectorStore()

# 1. Upsert document chunks with vector embeddings
upserted_count = vector_store.upsert_chunks(chunks, chunk_embeddings)
print(f"[OK] Successfully upserted {upserted_count} vectors into Pinecone index '{vector_store.index_name}'!")

# 2. Inspect index statistics
stats = vector_store.get_stats()
print(f"[x] Current Index Total Vectors: {stats.get('total_vector_count')}")

# 3. Perform semantic similarity search
test_query = "What is the policy on bereavement leave and immediate family members?"
test_vector = hf_client.embed_query(test_query)
search_results = vector_store.similarity_search(test_vector, top_k=3)

print(f"\n--- Top 3 Pinecone Matches for: '{test_query}' ---")
for idx, res in enumerate(search_results, 1):
    print(f"Match #{idx} [Score: {res['score']:.4f}, Source: {res['source']}, Page: {res['page_number']}]:")
    print(f"  {res['content'][:180]}...\n")


[OK] Successfully upserted 19 vectors into Pinecone index 'ragbot'!
[x] Current Index Total Vectors: 19

--- Top 3 Pinecone Matches for: 'What is the policy on bereavement leave and immediate family members?' ---
Match #1 [Score: 0.8170, Source: ABC_Company_Leave_Policy.pdf, Page: 2]:
  In the unfortunate event of the death of an immediate family member (spouse, child, parent, sibling, or
parent-in-law), employees are entitled to 5 working days of paid Bereavement...

Match #2 [Score: 0.6699, Source: ABC_Company_Leave_Policy.pdf, Page: 2]:
  leave may be availed prior to the expected date of delivery, with the remainder following childbirth. Employees must
notify HR and their Reporting Manager at least 8 weeks before t...

Match #3 [Score: 0.6575, Source: ABC_Company_Leave_Policy.pdf, Page: 3]:
  availed within 60 days of accrual, failing which it will lapse; it is not encashable.
4.9 Leave Without Pay (LWP)
Leave Without Pay may be granted at the sole discretion of managem...


---
## Section 4: Standard RAG Pipeline with Groq LLM (`app.rag`)

`app.rag.StandardRAG` unites the vector store, embeddings, and Groq's high-speed inference engine (`qwen/qwen3.8-27b`):
1. Takes the user's question.
2. Retrieves top-$k$ relevant chunks from Pinecone.
3. Formats a context-grounded prompt requiring strict policy adherence.
4. Synthesizes an answer with citations to source files and page numbers.


In [5]:
# ==============================================================================
# Cell 5: Standard RAG Execution with Groq
# ==============================================================================
standard_rag = StandardRAG(
    vector_store=vector_store,
    embedding_client=hf_client
)

question_1 = "How many days of casual leave do employees receive and how is it accrued?"
response_1 = standard_rag.ask(question_1, top_k=3)

print("==================================================")
print(f"QUESTION: {response_1.question}")
print("==================================================")
print("ANSWER:")
print(response_1.answer)
print("\nCITATIONS:")
for c in response_1.citations:
    print(f"  * [{c.source}, Page {c.page_number}]: {c.snippet[:100]}...")
print(f"Model Used: {response_1.model_used}")
print("==================================================")


QUESTION: How many days of casual leave do employees receive and how is it accrued?
ANSWER:
Employees receive a total of **12 days** of Casual Leave per year. This leave is accrued at a rate of **1 day per completed month of service** [Source: ABC_Company_Leave_Policy.pdf, Page: 2].

CITATIONS:
  * [ABC_Company_Leave_Policy.pdf, Page 2]: errands. Employees accrue 1 day of Casual Leave per completed month of service, up to a maximum of 1...
  * [ABC_Company_Leave_Policy.pdf, Page 2]: Leave Type
Annual Entitlement
Eligibility
Carry Forward
Marriage Leave
5 working days
Confirmed empl...
  * [ABC_Company_Leave_Policy.pdf, Page 1]: G
Misrepresentation of facts to avail any leave category will be treated as a violation of the Compa...
Model Used: qwen/qwen3.8-27b


---
## Section 5: Improving RAG with Query Rewriting & Evidence Grading (`app.advanced_rag`)

Real enterprise users rarely formulate perfect search queries. Standard RAG can suffer from:
- **Colloquial / Ambiguous queries**: The user asks *"can i take off if my kid is sick?"*, but the policy calls it *"Emergency Dependent Care"* or *"Sick Leave"*.
- **Irrelevant context noise**: Cosine search can retrieve tangentially related chunks that pollute the LLM's context.

`app.advanced_rag.AdvancedRAG` introduces two crucial improvements:
1. **Query Rewriter**: Translates informal employee questions into search-optimized semantic keywords before retrieval.
2. **Evidence Grader**: Evaluates each retrieved chunk using LLM classification, filtering out non-relevant noise before synthesis.
3. **Strict Grounding**: Synthesizes verified evidence with mandatory citation tags.


In [6]:
# ==============================================================================
# Cell 6: Inspecting Query Rewriting & Evidence Grading Individually
# ==============================================================================
groq_client = GroqLLMClient()
rewriter = QueryRewriter(groq_client)
grader = EvidenceGrader(groq_client)

informal_question = "can i take leave if someone in my family passes away?"

# 1. Test Query Rewriting
optimized_query = rewriter.rewrite(informal_question)
print(f"Original User Question: '{informal_question}'")
print(f"Rewritten Search Query : '{optimized_query}'")

# 2. Test Evidence Grading on search results
test_vec = hf_client.embed_query(optimized_query)
candidate_chunks = vector_store.similarity_search(test_vec, top_k=4)

print(f"\n--- Evidence Grading on {len(candidate_chunks)} Candidate Chunks ---")
for idx, chunk in enumerate(candidate_chunks, 1):
    grade = grader.grade(informal_question, chunk['content'])
    print(f"Chunk #{idx} (Page {chunk['page_number']}) -> Relevant: {grade.is_relevant} (Reason: {grade.reason})")


Original User Question: 'can i take leave if someone in my family passes away?'
Rewritten Search Query : 'Bereavement Leave Policy Eligibility Family Death'

--- Evidence Grading on 4 Candidate Chunks ---
Chunk #1 (Page 2) -> Relevant: True (Reason: The chunk explicitly defines Bereavement Leave for the death of an immediate family member, directly answering the inquiry about taking leave in such an event.)
Chunk #2 (Page 2) -> Relevant: True (Reason: The chunk contains section 4.6 'Bereavement Leave', which directly addresses the policy regarding leave for the death of an immediate family member.)
Chunk #3 (Page 3) -> Relevant: True (Reason: The chunk explicitly mentions 'Bereavement Leave' in the context of leave application procedures, which directly addresses the employee's inquiry about taking leave for a family member's passing.)
Chunk #4 (Page 1) -> Relevant: True (Reason: The chunk explicitly lists 'Bereavement Leave' with 5 working days for all employees, which directly addres

In [7]:
# ==============================================================================
# Cell 7: End-to-End Improving RAG Execution
# ==============================================================================
adv_rag = AdvancedRAG(
    vector_store=vector_store,
    embedding_client=hf_client,
    llm_client=groq_client
)

question_2 = "can i take leave if someone in my family passes away?"
response_2 = adv_rag.ask(question_2, top_k=5)

print("==================================================")
print(f"ORIGINAL QUESTION : {response_2.question}")
print(f"REWRITTEN QUERY   : {response_2.rewritten_query}")
print(f"CHUNKS FILTERED   : {response_2.relevant_chunks_count} relevant out of {response_2.retrieved_chunks_count} retrieved")
print("==================================================")
print("SYNTHESIZED ANSWER:")
print(response_2.answer)
print("\nVERIFIED CITATIONS:")
for cit in response_2.citations:
    print(f"  * [Source: {cit.source}, Page: {cit.page_number}]")
print("==================================================")


ORIGINAL QUESTION : can i take leave if someone in my family passes away?
REWRITTEN QUERY   : Bereavement Leave Policy Eligibility Family Death
CHUNKS FILTERED   : 4 relevant out of 5 retrieved
SYNTHESIZED ANSWER:
Yes, you are entitled to take leave if an immediate family member passes away. Employees are eligible for 5 working days of paid Bereavement Leave in the event of the death of an immediate family member, which includes a spouse, child, parent, sibling, or parent-in-law [Source: ABC_Company_Leave_Policy.pdf, Page: 2].

This leave can be taken immediately upon notifying your Reporting Manager, and it does not require the standard advance notice [Source: ABC_Company_Leave_Policy.pdf, Page: 2]. Additionally, Bereavement Leave is an exception to the general rule that all leave requests must be submitted in advance [Source: ABC_Company_Leave_Policy.pdf, Page: 3].

VERIFIED CITATIONS:
  * [Source: ABC_Company_Leave_Policy.pdf, Page: 2]
  * [Source: ABC_Company_Leave_Policy.pdf, Page

---
## Architecture Summary & Ready for Production

### Key Accomplishments:
1. **Modular Codebase**: All logic extracted into clean Python modules under `app/` (`config.py`, `schemas.py`, `ingestion/`, `embeddings.py`, `vectorstore.py`, `rag.py`, `advanced_rag.py`).
2. **Production Embeddings**: Hugging Face Inference API client producing 1024-dim vectors matching Pinecone.
3. **Vector Database**: Connected and indexing directly into Pinecone index `ragbot`.
4. **Groq High-Speed LLM**: Sub-second answers powered by `qwen/qwen3.8-27b`.
5. **Advanced RAG Guardrails**: Query Rewriting and Evidence Grading eliminating hallucinations and irrelevant context.
